# 04 - Workday Pattern Analysis

## Objective

This notebook analyzes regular `WORKDAY` attendance and work-context patterns.

The objective is to understand how employees are marked on normal workdays across different contexts such as:

- PRESENT
- Home
- Client Office
- Wipro Office
- Full Day Leave
- Half Day Leave
- Training
- Business Travel
- Other remarks

This analysis helps establish the normal workday baseline and provides context for the non-working day PRESENT pattern analyzed in Notebook 03.

No compliance conclusion is made at this stage. The goal is exploratory workforce pattern analysis.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

project_root = Path.cwd().parent
processed_path = project_root / "data" / "processed"

no_duplicate_file_path = processed_path / "combined_no_exact_duplicates.csv"

df = pd.read_csv(no_duplicate_file_path, low_memory=False)

print("Data loaded successfully")
print("Shape:", df.shape)

In [ ]:
workday_df = df[df["WEEKOFF_HOLIDAY"] == "WORKDAY"].copy()

print("Total records:", df.shape[0])
print("WORKDAY records:", workday_df.shape[0])



## WORKDAY Remarks Distribution

This section analyzes the distribution of REMARKS for regular WORKDAY records.

The goal is to understand the main work-context categories on normal workdays, such as PRESENT, Home, Client Office, Wipro Office, Leave, Training, and Business Travel.

This provides the baseline workday behavior before analyzing campus hours or location-level patterns.

In [ ]:
workday_remarks_distribution = (
    workday_df["REMARKS"]
    .value_counts(dropna=False)
    .reset_index()
)

workday_remarks_distribution.columns = ["REMARKS", "workday_record_count"]

workday_remarks_distribution.head(20)

## WORKDAY Remarks Distribution Interpretation

The WORKDAY remarks distribution shows that normal workdays include multiple work-context categories.

Top categories are:

- PRESENT: 2,032,796 records (~40.34%)
- Home: 1,840,361 records (~36.52%)
- Client Office: 461,298 records (~9.15%)

This indicates that WORKDAY records are not simply office-present or absent. They include multiple valid work contexts such as campus presence, home/remote work, client office, leave, and Wipro office.


## Workday Remark Bucketing

The raw REMARKS column contains many detailed work-context values. To make WORKDAY analysis easier to interpret, remarks are grouped into broader work-mode buckets.

The buckets are exploratory and should be validated with business stakeholders before being used for final compliance or policy interpretation.

Proposed buckets:

- Campus / Office Presence
- Home / Remote
- Client Office
- Leave
- Business Exception
- Mixed Context
- Unknown / Unclear

In [ ]:
def classify_workday_remark(remark):
    """
    Classifies raw REMARKS values into broader workday work-mode buckets.
    """
    
    if pd.isna(remark):
        return "Unknown / Unclear"
    
    remark = str(remark).strip()
    
    # Unknown or unclear values
    if remark == "-":
        return "Unknown / Unclear"
    
    # Mixed context values
    # Example: Home AND Wipro Office, Client Office AND Home
    if " AND " in remark:
        return "Mixed Context"
    
    # Campus / Office presence
    if remark in ["PRESENT", "Wipro Office", "TEMP CARD"]:
        return "Campus / Office Presence"
    
    # Home / Remote
    if remark == "Home":
        return "Home / Remote"
    
    # Client Office
    if remark == "Client Office":
        return "Client Office"
    
    # Leave-related
    if remark in ["FULL DAY LEAVE", "HALF DAY LEAVE", "LEAVE AND OOD"]:
        return "Leave"
    
    # Business exception / other valid work context
    if remark in [
        "Training",
        "Business Travel",
        "External Conference/Meeting",
        "FULL DAY OOD",
        "TRAVEL"
    ]:
        return "Business Exception"
    
    # Fallback bucket
    return "Other / Review"

In [ ]:
workday_df["workday_remark_bucket"] = workday_df["REMARKS"].apply(classify_workday_remark)

workday_bucket_distribution = (
    workday_df["workday_remark_bucket"]
    .value_counts(dropna=False)
    .reset_index()
)

workday_bucket_distribution.columns = [
    "workday_remark_bucket",
    "workday_record_count"
]

workday_bucket_distribution

In [ ]:
workday_bucket_distribution["workday_record_pct"] = (
    workday_bucket_distribution["workday_record_count"] /
    workday_df.shape[0] * 100
)

workday_bucket_distribution

## Workday Remark Bucket Interpretation

After grouping raw WORKDAY remarks into broader work-mode buckets, the distribution shows that regular workdays are not binary office/absence records.

Key bucket-level findings:

- Campus / Office Presence: 2,216,965 records (~44.00%)
- Home / Remote: 1,840,361 records (~36.52%)
- Client Office: 461,298 records (~9.15%)
- Leave: 364,370 records (~7.23%)
- Mixed Context: 80,406 records (~1.60%)
- Unknown / Unclear: 59,078 records (~1.17%)
- Business Exception: 16,563 records (~0.33%)

This indicates that WORKDAY records contain multiple valid work contexts such as campus presence, home/remote work, client office, leave, mixed work modes, and business exceptions.

The high share of Home / Remote records suggests that remote or no-campus work context is a major part of workday behavior. However, this should not be interpreted as non-compliance without business validation.

Client Office records should also be interpreted carefully because employees may be working from client locations where Wipro campus TIC may be zero.

Leave records should be handled separately in campus-hour analysis because including leave records may distort average workday TIC values.

## TIC Hours by Workday Remark Bucket

This section compares TIC hours across the broader workday remark buckets.

The goal is to validate whether the bucket behavior aligns with business expectations.

For example:

- Campus / Office Presence is expected to have higher TIC hours.
- Home / Remote is expected to have low or zero TIC hours.
- Client Office may have low TIC because the employee may be working outside Wipro campus.
- Leave records should generally have low or zero TIC.
- Mixed Context and Unknown categories require cautious interpretation.

In [ ]:
workday_bucket_tic_summary = (
    workday_df
    .groupby("workday_remark_bucket")["TIC_HOURS_NEW"]
    .describe()
    .reset_index()
)

workday_bucket_tic_summary

## TIC Hours by Workday Bucket Interpretation

The TIC hour distribution by workday remark bucket shows that the bucket classification is broadly aligned with expected business behavior.

Key observations:

- Campus / Office Presence has the highest TIC values:
  - Mean TIC: approximately 6.16 hours
  - Median TIC: approximately 6.80 hours
  - 75th percentile TIC: approximately 9.46 hours

This confirms that the Campus / Office Presence bucket represents meaningful campus presence.

- Home / Remote has very low TIC values:
  - Mean TIC: approximately 0.13 hours
  - Median TIC: 0 hours

This is expected because Home / Remote records generally indicate no campus presence.

- Client Office also has low TIC values:
  - Mean TIC: approximately 0.31 hours
  - Median TIC: 0 hours

This should not be treated as absence because Client Office may represent valid work outside Wipro campus.

- Leave records have near-zero TIC values, which is expected. However, some leave records show positive maximum TIC values, so such cases may require separate validation.

Overall, the TIC distribution supports the workday bucket classification. Campus / Office Presence behaves differently from Home / Remote, Client Office, and Leave, which confirms that workday records should be analyzed by work-context buckets rather than using a simple present/absent interpretation.

## TIC Segmentation within Campus / Office Presence Records

After identifying Campus / Office Presence as the largest WORKDAY bucket, this section further segments those records based on TIC_HOURS_NEW.

The objective is to understand whether campus/office presence records represent no campus time, short visits, substantial campus presence, or very high campus presence.

Exploratory segments:

- No Campus Time: TIC = 0
- Short Campus Presence: 0 < TIC < 6
- Substantial Campus Presence: TIC >= 6
- Very High Campus Presence: TIC > 12

These thresholds are exploratory and should be validated with business policy before being used for compliance conclusions.

In [ ]:
# Filter only Campus / Office Presence workday records
campus_office_workday = workday_df[
    workday_df["workday_remark_bucket"] == "Campus / Office Presence"
].copy()

print("Campus / Office Presence records:", campus_office_workday.shape[0])


# Create TIC-based campus presence segments
conditions = [
    campus_office_workday["TIC_HOURS_NEW"] == 0,
    (campus_office_workday["TIC_HOURS_NEW"] > 0) & (campus_office_workday["TIC_HOURS_NEW"] < 6),
    (campus_office_workday["TIC_HOURS_NEW"] >= 6) & (campus_office_workday["TIC_HOURS_NEW"] <= 12),
    campus_office_workday["TIC_HOURS_NEW"] > 12
]

choices = [
    "No Campus Time",
    "Short Campus Presence",
    "Substantial Campus Presence",
    "Very High Campus Presence"
]

campus_office_workday["tic_presence_segment"] = np.select(
    conditions,
    choices,
    default="Unknown"
)


# Segment summary
campus_tic_segment_summary = (
    campus_office_workday["tic_presence_segment"]
    .value_counts()
    .reset_index()
)

campus_tic_segment_summary.columns = [
    "tic_presence_segment",
    "record_count"
]

campus_tic_segment_summary["record_pct"] = (
    campus_tic_segment_summary["record_count"] /
    campus_office_workday.shape[0] * 100
)

campus_tic_segment_summary

## TIC Segmentation Interpretation within Campus / Office Presence

Results:

- Substantial Campus Presence: 1,183,488 records (~53.38%)
- Short Campus Presence: 800,756 records (~36.12%)
- No Campus Time: 177,785 records (~8.02%)
- Very High Campus Presence: 54,936 records (~2.48%)

This shows that more than half of Campus / Office Presence records have substantial campus time between 6 and 12 hours.

Short campus presence is also a meaningful portion, indicating that not all campus/office records represent full-day presence.

Very High Campus Presence records are relatively rare within this bucket and should be treated as review cases rather than direct evidence of overwork.

The 2.48% figure is calculated within the Campus / Office Presence bucket, not across all WORKDAY records.

## Location-wise Workday Bucket Analysis

After understanding the overall WORKDAY bucket distribution, the next step is to analyze how these work-context buckets vary by location.

The objective is to identify whether certain locations are more Campus / Office Presence-heavy, Home / Remote-heavy, Client Office-heavy, or Leave-heavy.

This helps build a location-level workday baseline and provides context for future employee-level and predictive analysis.

No compliance conclusion is made at this stage. These are exploratory workforce pattern insights.

In [ ]:
# Count records by LOCATION and workday_remark_bucket
location_workday_bucket = (
    workday_df
    .groupby(["LOCATION", "workday_remark_bucket"])
    .size()
    .reset_index(name="bucket_record_count")
)

# Total WORKDAY records by location
location_workday_total = (
    workday_df
    .groupby("LOCATION")
    .size()
    .reset_index(name="location_total_workday_records")
)

# Merge bucket counts with location totals
location_workday_bucket = location_workday_bucket.merge(
    location_workday_total,
    on="LOCATION",
    how="left"
)

# Calculate bucket rate within each location
location_workday_bucket["bucket_rate_pct"] = (
    location_workday_bucket["bucket_record_count"] /
    location_workday_bucket["location_total_workday_records"] * 100
)



In [ ]:
location_workday_bucket_pivot = (
    location_workday_bucket
    .pivot_table(
        index="LOCATION",
        columns="workday_remark_bucket",
        values="bucket_rate_pct",
        fill_value=0
    )
    .reset_index()
)



In [ ]:
major_locations = [
    "BENGALURU",
    "PUNE",
    "HYDERABAD",
    "MUMBAI",
    "KOLKATA",
    "CHENNAI",
    "COIMBATORE",
    "MYSORE",
    "KOCHI",
    "GREATER NOIDA",
    "BHUBANESWAR"
]

major_location_workday_bucket_pivot = location_workday_bucket_pivot[
    location_workday_bucket_pivot["LOCATION"].isin(major_locations)
].copy()
major_location_workday_bucket_pivot

## Location-wise Workday Bucket Insight

The location-wise workday bucket analysis shows that regular workday behavior varies significantly across locations.

Mysore is strongly campus-heavy, with around 73.47% of WORKDAY records falling under Campus / Office Presence. Kolkata and Bhubaneswar also show relatively high Campus / Office Presence shares.

Mumbai stands out because of its high Client Office share of around 19.70%, suggesting a stronger client-location work context compared with other major locations. This should not be interpreted as absence, because Client Office may represent valid work outside Wipro campus.

Bengaluru has the highest overall data volume but its workday bucket distribution is close to the overall baseline, with Campus / Office Presence around 42.56% and Home / Remote around 37.43%.

Greater Noida shows a relatively higher Home / Remote share, while Hyderabad shows a balanced Campus/Home pattern with a meaningful Client Office component.

Overall, this confirms that WORKDAY behavior is location-dependent. Therefore, workday analysis should not apply one overall interpretation across all locations. Each major location should be interpreted based on its own work-mode mix.

## Chart: Location-wise Workday Bucket Mix

This chart compares the major workday bucket distribution across selected major locations.

The chart focuses on the four main buckets:

- Campus / Office Presence
- Home / Remote
- Client Office
- Leave

The purpose is to visually compare how regular workday behavior differs by location.

In [ ]:
import matplotlib.pyplot as plt

# Select major buckets for cleaner visualization
major_buckets = [
    "Campus / Office Presence",
    "Home / Remote",
    "Client Office",
    "Leave"
]

# Prepare chart data
chart_data = (
    major_location_workday_bucket_pivot
    .set_index("LOCATION")[major_buckets]
)

# Optional: sort by Campus / Office Presence rate
chart_data = chart_data.sort_values("Campus / Office Presence", ascending=False)

# Plot stacked bar chart
plt.figure(figsize=(14, 7))

chart_data.plot(
    kind="bar",
    stacked=True,
    figsize=(14, 7)
)

plt.title("Location-wise Workday Bucket Mix")
plt.xlabel("Location")
plt.ylabel("Percentage of WORKDAY Records")
plt.xticks(rotation=45, ha="right")
plt.legend(title="Workday Bucket", bbox_to_anchor=(1.05, 1), loc="upper left")

plt.tight_layout()
plt.show()

### Interpretation of Location-wise Workday Bucket Mix

The stacked bar chart shows that regular WORKDAY behavior varies significantly across major locations.

Mysore is the most campus-heavy location among the selected locations, with Campus / Office Presence forming the majority of its WORKDAY records.

Kolkata and Bhubaneswar also show relatively high Campus / Office Presence shares, indicating stronger campus-oriented workday behavior.

Mumbai stands out because of its high Client Office share. This suggests that Mumbai has a stronger client-location work context compared with other major locations. Client Office should not be interpreted as absence because employees may be working from client locations where Wipro campus TIC may be zero.

Bengaluru shows a more balanced workday mix, with both Campus / Office Presence and Home / Remote forming large portions of its WORKDAY records.

Greater Noida shows a relatively higher Home / Remote share, suggesting a stronger remote/no-campus workday pattern compared with other major locations.

Overall, this confirms that WORKDAY behavior is location-dependent. Therefore, workday analysis should not apply one single interpretation across all locations.

## Location-wise TIC Segmentation within Campus / Office Presence

This section analyzes TIC intensity within the Campus / Office Presence bucket across major locations.

The objective is to understand whether campus/office records represent short visits, substantial campus presence, or very high campus presence in each location.

Exploratory TIC segments:

- No Campus Time: TIC = 0
- Short Campus Presence: 0 < TIC < 6
- Substantial Campus Presence: 6 <= TIC <= 12
- Very High Campus Presence: TIC > 12

These thresholds are exploratory and should not be treated as compliance rules without business validation.


In [ ]:
# Ensure Campus / Office Presence records exist
campus_office_workday = workday_df[
    workday_df["workday_remark_bucket"] == "Campus / Office Presence"
].copy()

# Recreate TIC presence segment if needed
conditions = [
    campus_office_workday["TIC_HOURS_NEW"] == 0,
    (campus_office_workday["TIC_HOURS_NEW"] > 0) & (campus_office_workday["TIC_HOURS_NEW"] < 6),
    (campus_office_workday["TIC_HOURS_NEW"] >= 6) & (campus_office_workday["TIC_HOURS_NEW"] <= 12),
    campus_office_workday["TIC_HOURS_NEW"] > 12
]

choices = [
    "No Campus Time",
    "Short Campus Presence",
    "Substantial Campus Presence",
    "Very High Campus Presence"
]

campus_office_workday["tic_presence_segment"] = np.select(
    conditions,
    choices,
    default="Unknown"
)

# Location-wise TIC segment count
location_campus_tic_segment = (
    campus_office_workday
    .groupby(["LOCATION", "tic_presence_segment"])
    .size()
    .reset_index(name="segment_record_count")
)

# Total Campus / Office Presence records by location
location_campus_total = (
    campus_office_workday
    .groupby("LOCATION")
    .size()
    .reset_index(name="location_campus_office_records")
)

# Merge and calculate rate
location_campus_tic_segment = location_campus_tic_segment.merge(
    location_campus_total,
    on="LOCATION",
    how="left"
)

location_campus_tic_segment["segment_rate_pct"] = (
    location_campus_tic_segment["segment_record_count"] /
    location_campus_tic_segment["location_campus_office_records"] * 100
)



In [ ]:
location_campus_tic_pivot = (
    location_campus_tic_segment
    .pivot_table(
        index="LOCATION",
        columns="tic_presence_segment",
        values="segment_rate_pct",
        fill_value=0
    )
    .reset_index()
)

major_location_campus_tic_pivot = location_campus_tic_pivot[
    location_campus_tic_pivot["LOCATION"].isin(major_locations)
].copy()

major_location_campus_tic_pivot

In [ ]:
import matplotlib.pyplot as plt

# Select segment columns
tic_segment_columns = [
    "No Campus Time",
    "Short Campus Presence",
    "Substantial Campus Presence",
    "Very High Campus Presence"
]

# Prepare chart data
chart_data = (
    major_location_campus_tic_pivot
    .set_index("LOCATION")[tic_segment_columns]
)

# Sort by Substantial Campus Presence for clearer comparison
chart_data = chart_data.sort_values(
    "Substantial Campus Presence",
    ascending=False
)

# Plot stacked bar chart
chart_data.plot(
    kind="bar",
    stacked=True,
    figsize=(14, 7)
)

plt.title("Location-wise TIC Segmentation within Campus / Office Presence")
plt.xlabel("Location")
plt.ylabel("Percentage of Campus / Office Presence Records")
plt.xticks(rotation=45, ha="right")
plt.legend(
    title="TIC Presence Segment",
    bbox_to_anchor=(1.05, 1),
    loc="upper left"
)

plt.tight_layout()
plt.show()

### Interpretation of Location-wise TIC Segmentation within Campus / Office Presence

The TIC segmentation chart shows that Campus / Office Presence records have different intensity patterns across locations.

Mumbai and Kolkata show the strongest substantial campus presence. A large majority of their Campus / Office Presence records fall in the 6–12 hour TIC range, and both locations also have a noticeable share of Very High Campus Presence records. This supports the earlier observation that Mumbai and Kolkata show stronger full-day or near full-day presence patterns.

Mysore shows a very different pattern. Although Mysore appeared campus-heavy in the workday bucket analysis, most of its Campus / Office Presence records fall under Short Campus Presence. This indicates that Mysore is campus-heavy by work-context bucket, but mostly short-duration by TIC intensity.

Bengaluru and Pune show mixed campus intensity, with both Short Campus Presence and Substantial Campus Presence contributing meaningful shares.

Greater Noida has a relatively higher share of No Campus Time and Short Campus Presence within the Campus / Office bucket, so this location should be interpreted cautiously.

Overall, this chart confirms that location-level workday behavior should be analyzed using both remark buckets and TIC intensity. A location may appear campus-heavy based on remarks, but TIC segmentation is needed to understand whether that presence represents short visits or substantial campus time.'
''"

Mumbai and Kolkata show strong full-day workday campus intensity, Mysore shows mostly short-duration campus presence, and Bengaluru/Pune show mixed campus intensity.

## Employee-level Workday Behavior Summary

After analyzing WORKDAY behavior at overall and location levels, this section summarizes workday behavior at employee level.

The objective is to create employee-level workday features such as:

- Total WORKDAY records
- Campus / Office Presence records
- Home / Remote records
- Client Office records
- Leave records
- Average WORKDAY TIC hours
- Workday 6+ TIC records
- Short campus presence records
- Substantial campus presence records
- Very high campus presence records

These features will help support future predictive or risk-scoring analysis.

No compliance conclusion is made at this stage. The purpose is exploratory feature creation.
`

In [ ]:
employee_workday_summary = (
    workday_df
    .groupby("EMP_CODE")
    .agg(
        total_workday_records=("ATTENDANCE_DATE", "count"),
        unique_workday_dates=("ATTENDANCE_DATE", "nunique"),
        
        campus_office_records=(
            "workday_remark_bucket",
            lambda x: (x == "Campus / Office Presence").sum()
        ),
        
        home_remote_records=(
            "workday_remark_bucket",
            lambda x: (x == "Home / Remote").sum()
        ),
        
        client_office_records=(
            "workday_remark_bucket",
            lambda x: (x == "Client Office").sum()
        ),
        
        leave_records=(
            "workday_remark_bucket",
            lambda x: (x == "Leave").sum()
        ),
        
        mixed_context_records=(
            "workday_remark_bucket",
            lambda x: (x == "Mixed Context").sum()
        ),
        
        business_exception_records=(
            "workday_remark_bucket",
            lambda x: (x == "Business Exception").sum()
        ),
        
        unknown_unclear_records=(
            "workday_remark_bucket",
            lambda x: (x == "Unknown / Unclear").sum()
        ),
        
        avg_workday_tic_hours=("TIC_HOURS_NEW", "mean"),
        median_workday_tic_hours=("TIC_HOURS_NEW", "median"),
        max_workday_tic_hours=("TIC_HOURS_NEW", "max"),
        
        workday_6plus_tic_records=(
            "TIC_HOURS_NEW",
            lambda x: (x >= 6).sum()
        ),
        
        workday_12plus_tic_records=(
            "TIC_HOURS_NEW",
            lambda x: (x > 12).sum()
        )
    )
    .reset_index()
)




In [ ]:
print("Unique employees in WORKDAY summary:", employee_workday_summary.shape[0])

employee_workday_summary[
    [
        "total_workday_records",
        "unique_workday_dates",
        "campus_office_records",
        "home_remote_records",
        "client_office_records",
        "leave_records",
        "avg_workday_tic_hours",
        "workday_6plus_tic_records",
        "workday_12plus_tic_records"
    ]
].describe()

## Employee-level Workday Summary Interpretation

The employee-level workday summary shows that 114,557 employees have WORKDAY records in the dataset.

Most employees have around 42 unique WORKDAY dates, which suggests that the dataset covers a meaningful workday period for employee-level behavior analysis.

At an employee level, workday behavior appears mixed:

- The median employee has around 19 Campus / Office Presence records.
- The median employee has around 17 Home / Remote records.
- Client Office records are concentrated among fewer employees, since the median is 0.
- The median employee has 5 records with TIC >= 6 hours.
- Very high TIC records above 12 hours are rare for most employees, since the median and 75th percentile are both 0.

This suggests that workday behavior is not uniform across employees. Some employees are more campus-oriented, some are more home/remote-oriented, and Client Office behavior appears concentrated among specific employees.

However, this summary is still based on record-level counts. Since employees may have multiple project-level records for the same date, employee-date level aggregation should be used before creating final prediction features.

In [ ]:
employee_workday_summary["campus_office_rate_pct"] = (
    employee_workday_summary["campus_office_records"] /
    employee_workday_summary["total_workday_records"] * 100
)

employee_workday_summary["home_remote_rate_pct"] = (
    employee_workday_summary["home_remote_records"] /
    employee_workday_summary["total_workday_records"] * 100
)

employee_workday_summary["client_office_rate_pct"] = (
    employee_workday_summary["client_office_records"] /
    employee_workday_summary["total_workday_records"] * 100
)

employee_workday_summary["leave_rate_pct"] = (
    employee_workday_summary["leave_records"] /
    employee_workday_summary["total_workday_records"] * 100
)

employee_workday_summary["workday_6plus_tic_rate_pct"] = (
    employee_workday_summary["workday_6plus_tic_records"] /
    employee_workday_summary["total_workday_records"] * 100
)

employee_workday_summary["workday_12plus_tic_rate_pct"] = (
    employee_workday_summary["workday_12plus_tic_records"] /
    employee_workday_summary["total_workday_records"] * 100
)



In [ ]:
employee_workday_summary[
    [
        "campus_office_rate_pct",
        "home_remote_rate_pct",
        "client_office_rate_pct",
        "leave_rate_pct",
        "workday_6plus_tic_rate_pct",
        "workday_12plus_tic_rate_pct"
    ]
].describe()

## Employee-level Workday Rate Interpretation

Employee-level rate analysis shows that workday behavior varies significantly across employees.

Key observations:

- The median Campus / Office Presence rate is approximately 45.33%, indicating that a typical employee has a mixed workday pattern rather than being fully campus-based.
- The median Home / Remote rate is approximately 39.53%, confirming that Home / Remote is a major workday context at employee level.
- Client Office behavior appears concentrated among specific employees. The median and 75th percentile Client Office rate are both 0%, but the mean is 9.32%, suggesting that a smaller group of employees contributes heavily to Client Office records.
- The median Leave rate is approximately 4.76%, indicating that leave is present but not dominant for most employees.
- The median 6+ TIC rate is approximately 12.82%, while the 75th percentile is 45.24%. This shows that substantial campus presence is unevenly distributed across employees.
- The 12+ TIC rate is 0% for most employees, indicating that very high campus presence is rare and should be treated as a review feature rather than direct evidence of overwork.

Overall, employee-level workday behavior is mixed and highly variable. Some employees are campus-heavy, some are home/remote-heavy, some are client-office-heavy, and some show mixed workday patterns.

## Employee Workday Behavior Segmentation

Employee-level workday rates provide useful feature-level information, but it is easier to interpret workforce behavior by grouping employees into broader behavioral segments.

This section classifies employees into exploratory workday behavior segments based on the dominant work-context observed in their WORKDAY records.

Segments include:

- Campus-heavy
- Home/Remote-heavy
- Client Office-heavy
- Leave-heavy
- Mixed Workday Pattern

These segments are exploratory analytical categories and should not be interpreted as compliance, productivity, or performance indicators.

In [ ]:
def classify_workday_behavior(row):

    if row["leave_rate_pct"] >= 30:
        return "Leave-heavy"

    elif row["client_office_rate_pct"] >= 40:
        return "Client Office-heavy"

    elif row["campus_office_rate_pct"] >= 60:
        return "Campus-heavy"

    elif row["home_remote_rate_pct"] >= 60:
        return "Home/Remote-heavy"

    else:
        return "Mixed Workday Pattern"


employee_workday_summary["workday_behavior_segment"] = (
    employee_workday_summary.apply(
        classify_workday_behavior,
        axis=1
    )
)

In [ ]:
employee_workday_segment_summary = (
    employee_workday_summary["workday_behavior_segment"]
    .value_counts()
    .reset_index()
)

employee_workday_segment_summary.columns = [
    "workday_behavior_segment",
    "employee_count"
]

employee_workday_segment_summary["employee_pct"] = (
    employee_workday_segment_summary["employee_count"]
    / employee_workday_summary.shape[0]
    * 100
)

employee_workday_segment_summary

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))

plt.bar(
    employee_workday_segment_summary["workday_behavior_segment"],
    employee_workday_segment_summary["employee_count"]
)

plt.title("Employee Workday Behavior Segments")
plt.xlabel("Workday Behavior Segment")
plt.ylabel("Employee Count")

plt.xticks(rotation=20)

plt.tight_layout()
plt.show()

## Employee Workday Behavior Segmentation Insight

Employee segmentation shows that the workforce is highly diverse in terms of normal workday behavior.

Key observations:

- Mixed Workday Pattern is the largest segment, representing approximately 46.15% of employees.
- Campus-heavy employees represent approximately 25.83% of employees.
- Home/Remote-heavy employees represent approximately 15.60% of employees.
- Client Office-heavy employees represent approximately 10.75% of employees, indicating that client-location work is concentrated within a smaller but meaningful employee group.
- Leave-heavy employees represent only 1.68% of employees.

Overall, the largest employee segment follows a mixed workday pattern rather than a single dominant work mode. This supports earlier findings that workforce behavior should be interpreted as a combination of Campus / Office Presence, Home / Remote, and Client Office contexts rather than a simple office-versus-remote classification.

These segments are exploratory behavioral categories and should not be interpreted as performance, compliance, or productivity measures.

## Day-of-Week Footfall Pattern Analysis

A natural extension of the WORKDAY analysis is future office footfall forecasting.

Before building forecasting models, it is important to understand whether campus attendance follows consistent weekday patterns.

This section analyzes average campus footfall across different weekdays using unique employee counts within the Campus / Office Presence bucket.

The goal is to identify recurring attendance patterns that may support future location-wise and organization-wide footfall forecasting.

In [ ]:
workday_df["ATTENDANCE_DATE"] = pd.to_datetime(
    workday_df["ATTENDANCE_DATE"]
)

workday_df["weekday_name"] = (
    workday_df["ATTENDANCE_DATE"]
    .dt.day_name()
)

In [ ]:
campus_workday_df = workday_df[
    workday_df["workday_remark_bucket"] == "Campus / Office Presence"
].copy()

In [ ]:
daily_footfall = (
    campus_workday_df
    .groupby("ATTENDANCE_DATE")["EMP_CODE"]
    .nunique()
    .reset_index(name="daily_campus_footfall")
)



In [ ]:
daily_footfall["weekday_name"] = (
    daily_footfall["ATTENDANCE_DATE"]
    .dt.day_name()
)

In [ ]:
weekday_footfall = (
    daily_footfall
    .groupby("weekday_name")["daily_campus_footfall"]
    .agg(
        average_footfall="mean",
        median_footfall="median",
        max_footfall="max",
        min_footfall="min"
    )
    .reset_index()
)

weekday_order = [
    "Monday",
    "Tuesday",
    "Wednesday",
    "Thursday",
    "Friday"
]

weekday_footfall["weekday_name"] = pd.Categorical(
    weekday_footfall["weekday_name"],
    categories=weekday_order,
    ordered=True
)

weekday_footfall = weekday_footfall.sort_values(
    "weekday_name"
)



In [ ]:
plot_df = weekday_footfall.copy()

plot_df = plot_df[
    plot_df["weekday_name"].notna()
].copy()



In [ ]:
import matplotlib.pyplot as plt

plot_df = weekday_footfall.dropna(
    subset=["weekday_name"]
).copy()

plot_df = plot_df.set_index("weekday_name")

plot_df = plot_df.loc[
    ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday"]
]

plt.figure(figsize=(10,6))

plt.plot(
    plot_df.index,
    plot_df["average_footfall"],
    marker="o",
    linewidth=3
)

for x, y in zip(
    plot_df.index,
    plot_df["average_footfall"]
):
    plt.text(
        x,
        y,
        f"{y:,.0f}",
        ha="center"
    )

plt.title("Average Campus Footfall by Weekday")
plt.xlabel("Weekday")
plt.ylabel("Average Daily Campus Footfall")

plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## Weekday-Based Footfall Pattern Insight

The weekday-level footfall analysis shows a clear attendance pattern across the workweek.

Key observations:

- Wednesday records the highest average campus footfall (~55,868 employees).
- Tuesday is the second-highest attendance day (~52,581 employees).
- Friday records the lowest average campus footfall (~37,459 employees).
- Monday and Thursday show moderate attendance levels compared with Tuesday and Wednesday.

The difference between Wednesday and Friday exceeds 18,000 employees, indicating that office attendance is strongly influenced by weekday effects.

This finding provides evidence that day-of-week should be included as an important predictive feature in future footfall forecasting models.

## Key Conclusions

1. WORKDAY records represent multiple valid work contexts rather than a simple present/absent model.

2. Campus / Office Presence and Home / Remote are the dominant workday modes.

3. TIC validation confirms that the workday bucket framework aligns with expected campus-presence behavior.

4. Workday behavior varies significantly across locations. Mumbai shows a strong Client Office pattern, while Mysore is strongly campus-oriented.

5. Employee workday behavior is highly diverse. Mixed Workday Pattern is the largest employee segment.

6. Campus attendance follows a clear weekday pattern, with Wednesday showing the highest average footfall and Friday the lowest.

7. These findings provide a strong foundation for future office footfall forecasting.

## Future Predictive Direction

The analyses performed in this notebook establish the foundation for future office footfall forecasting.

Several strong predictive signals were identified:

- Location-specific attendance behavior
- Employee workday behavior segments
- Campus presence intensity patterns
- Clear weekday attendance effects

A logical next step is to build daily and location-wise footfall forecasting models using historical campus attendance patterns.

Potential prediction outputs include:

- Future organization-wide office footfall
- Future location-wise office footfall
- Weekday-specific attendance forecasts
- Occupancy planning insights